# Notebook 02: Data Cleaning & Preprocessing (Chunked Pipeline)
Designed for ~20M rows.

In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATASET, PROCESSED_DATA_DIR

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATASET

PosixPath('/Users/subhankarbiswas/fifa-player-analytics/data/raw/players.csv')

In [13]:
import duckdb

con = duckdb.connect()

con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='8GB'")

In [14]:
con.sql(f"""
SELECT *
FROM read_csv_auto(
'{RAW_DATASET}',
sample_size=100000
)
LIMIT 5
""")

┌───────────┬──────────────────────────────────────────┬──────────────┬─────────────┬──────────────────┬────────────────┬────────────────────────────────┬──────────────────┬─────────┬───────────┬───────────┬──────────┬───────┬────────────┬───────────┬───────────┬───────────┬────────────────┬──────────────┬──────────────┬─────────────────────┬───────────────┬────────────────────┬──────────────────┬──────────────────┬────────────────────────────────┬────────────────┬──────────────────┬────────────────┬─────────────────┬──────────────────────┬────────────────┬───────────┬─────────────┬──────────────────────────┬───────────────┬──────────────────┬───────────┬────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬──────────┬─────────┬───────────┬───────────┬────────┬────────────────────┬────────────────────

In [15]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_csv_auto(
'{RAW_DATASET}',
sample_size=100000
)
""")

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ player_url       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fifa_version     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ fifa_update      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ fifa_update_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ short_name       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ long_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_positions │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ overall          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ potential        │ BIGI

In [16]:
output = PROCESSED_DATA_DIR / "players_clean"

con.execute(f"""

COPY (

SELECT *

FROM read_csv_auto(

'{RAW_DATASET}',

sample_size=-1,

ignore_errors=true,

union_by_name=true,

all_varchar=true

)

)

TO '{output}'

(

FORMAT PARQUET,

COMPRESSION ZSTD,

ROW_GROUP_SIZE 100000

)

""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [17]:
con.sql(f"""

SELECT COUNT(*)

FROM parquet_scan('{output}')

""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     10003590 │
└──────────────┘

In [18]:
con.sql(f"""

SELECT *

FROM parquet_scan('{output}')

USING SAMPLE 20 ROWS

""")

┌───────────┬───────────────────────────────────────────┬──────────────┬─────────────┬──────────────────┬────────────────┬─────────────────────────────┬──────────────────┬─────────┬───────────┬───────────┬──────────┬─────────┬────────────┬───────────┬───────────┬───────────┬────────────────┬──────────────┬──────────────┬──────────────────────┬───────────────┬────────────────────┬──────────────────┬──────────────────┬────────────────────────────────┬────────────────┬──────────────────┬────────────────┬─────────────────┬──────────────────────┬────────────────┬───────────┬─────────────┬──────────────────────────┬───────────────┬──────────────────┬───────────┬────────────────────┬─────────────┬────────────────────────────────────────────────────────────────────────────┬─────────┬──────────┬─────────┬───────────┬───────────┬─────────┬────────────────────┬─────────────────────┬────────────────────────────┬─────────────────────────┬───────────────────┬─────────────────┬─────────────┬─────────

In [19]:
con.sql(f"""

SELECT

COUNT(*) total_rows,

COUNT(value_eur) value_rows,

COUNT(club_name) club_rows

FROM parquet_scan('{output}')

""")

┌────────────┬────────────┬───────────┐
│ total_rows │ value_rows │ club_rows │
│   int64    │   int64    │   int64   │
├────────────┼────────────┼───────────┤
│   10003590 │    9870793 │   9887195 │
└────────────┴────────────┴───────────┘

In [20]:
schema = con.sql(f"""

DESCRIBE

SELECT *

FROM parquet_scan('{output}')

""").df()

schema.to_csv(
PROCESSED_DATA_DIR / "schema.csv",
index=False,
)

schema.head()

,column_name,column_type,null,key,default,extra
0,player_id,VARCHAR,YES,None,None,None
1,player_url,VARCHAR,YES,None,None,None
2,fifa_version,VARCHAR,YES,None,None,None
3,fifa_update,VARCHAR,YES,None,None,None
4,fifa_update_date,VARCHAR,YES,None,None,None
